In [2]:
# Reproducible dataset loading: download (via GitHub Release), verify SHA256, and load first 1M rows per project spec
import pooch
import pandas as pd

# Remote dataset (GitHub Release v1.0) and SHA256 for integrity verification
DATA_URL = "https://github.com/DrAlzahraniProjects/csusb_spring26_cse5140_team2/releases/download/v1.0/data.csv.gz"
DATA_HASH = "sha256:a56165ac7d7282a701e33a7c07ff6b3a9025f24c5bf84ce9462ab50f7ccd91cc"

# Download (if needed) and verify dataset checksum
file_path = pooch.retrieve(url=DATA_URL, known_hash=DATA_HASH)

# Per project spec: limit to first 1,000,000 rows
NROWS = 1_000_000

# Load compressed CSV (gzip) from verified path
df = pd.read_csv(file_path, nrows=NROWS, compression="gzip")

print("NROWS:", df.shape[0])

C:\Users\warre\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


NROWS: 1000000


In [3]:
print(df.columns.tolist())

['trip_id', 'year', 'month', 'week', 'day', 'hour', 'usertype', 'gender', 'starttime', 'stoptime', 'tripduration', 'temperature', 'events', 'from_station_id', 'from_station_name', 'latitude_start', 'longitude_start', 'dpcapacity_start', 'to_station_id', 'to_station_name', 'latitude_end', 'longitude_end', 'dpcapacity_end']


In [4]:
import numpy as np
import pandas as pd

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def build_features(df):

    X = pd.DataFrame(index=df.index)

    # Time features (already separated)
    X["year"] = df["year"]
    X["month"] = df["month"]
    X["week"] = df["week"]
    X["day"] = df["day"]
    X["hour"] = df["hour"]

    # Cyclic encoding for hour
    X["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    X["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    # Weather
    X["temperature"] = df["temperature"]

    # Distance between stations
    X["distance_km"] = haversine_km(
        df["latitude_start"],
        df["longitude_start"],
        df["latitude_end"],
        df["longitude_end"]
    )

    # Capacity difference
    X["capacity_diff"] = df["dpcapacity_end"] - df["dpcapacity_start"]

    # Categorical encoding
    X = pd.concat([
        X,
        pd.get_dummies(df["usertype"], prefix="usertype"),
        pd.get_dummies(df["gender"], prefix="gender"),
        pd.get_dummies(df["events"], prefix="event")
    ], axis=1)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    return X

In [5]:
X = build_features(df)
y = df["tripduration"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (1000000, 20)
Target shape: (1000000,)


In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% validation
    random_state=42     # reproducibility
)

print("Training size:", X_train.shape)
print("Validation size:", X_val.shape)

Training size: (800000, 20)
Validation size: (200000, 20)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

model = RandomForestRegressor(
    n_estimators=20,   # reduced
    max_depth=15,      # limited depth
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_val)

r2 = r2_score(y_val, predictions)
mape = mean_absolute_percentage_error(y_val, predictions)

print("Validation R²:", r2)
print("Validation MAPE:", mape)


Training shape: (800000, 20)
Validation shape: (200000, 20)
Validation R²: 0.6825383234477338
Validation MAPE: 0.2491724043799326


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# Increase model complexity for final evaluation:
# - n_estimators increased for more stable predictions
# - max_depth set to None to allow fully grown trees
# Note: This increases training time but may improve validation R²

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_val)

r2 = r2_score(y_val, predictions)
mape = mean_absolute_percentage_error(y_val, predictions)

print("Validation R²:", r2)
print("Validation MAPE:", mape)


Training shape: (800000, 20)
Validation shape: (200000, 20)
Validation R²: 0.677334156863189
Validation MAPE: 0.251039044956074
